In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [ ]:
from dotenv import load_dotenv

load_dotenv()

# Docking Workflow

This notebook demonstrates how to perform molecular docking using Deep Origin's drug discovery platform. You'll learn how to:

1. **Load and prepare proteins** - Load a protein structure and prepare it for docking
2. **Find binding pockets** - Identify potential binding sites on the protein
3. **Dock ligands** - Perform docking calculations for single or multiple ligands
4. **Monitor jobs** - Track the progress of docking calculations
5. **Analyze results** - Visualize and filter docking poses

Let's get started!


## Setup

First, we'll import the necessary Deep Origin drug discovery modules.


In [ ]:
from deeporigin.drug_discovery import (
    Complex,
    DATA_DIR,
    Protein,
    LigandSet,
)
import deeporigin

deeporigin.__version__

## Load Protein Structure

Here we load a protein structure from a PDB file. The `Complex` object represents a protein-ligand complex and will be used throughout the docking workflow. 




In [ ]:
protein = Protein.from_file(DATA_DIR / "brd" / "brd.pdb")
sim = Complex(protein=protein)
sim

## Load Ligands

Load a set of ligands from a CSV file containing SMILES strings. The `LigandSet` object allows you to work with multiple ligands at once. You can visualize them in a grid to see what molecules you're working with.


In [ ]:
ligands = LigandSet.from_csv(DATA_DIR / "ligands" / "smiles_to_dock.csv")
ligands

In [ ]:
ligands.show_grid()

## Assign Ligands to Complex

Associate the ligands with the protein complex. This prepares the system for docking calculations.


In [ ]:
sim.ligands = ligands
sim

## Visualize the Protein

Display the protein structure in 3D. This helps you understand the protein's structure before proceeding with docking.


In [ ]:
sim.protein.show()

## Prepare the Protein

Before docking, we need to prepare the protein structure. Water molecules are typically removed from crystal structures as they can interfere with docking calculations. 




In [ ]:
sim.protein.remove_water()

sim.protein.show()

## Find Pockets

The `find_pockets()` method of `Protein` uses computational methods to detect cavities and potential binding sites on the protein surface.

In [ ]:
pockets = sim.protein.find_pockets(pocket_count=1)
sim.protein.show(pockets=pockets)

## Inspect Binding Pockets

View the detected binding pockets. Each pocket represents a potential binding site. You'll typically want to dock ligands into the most promising pocket (often the largest or most druggable one).


In [ ]:
pockets

## Bulk Docking Workflow

For drug discovery, you'll often want to dock many ligands at once. The bulk docking workflow allows you to:

1. **Submit multiple docking jobs** - Dock all ligands in your ligand set
2. **Monitor progress** - Track job status in real-time
3. **Retrieve results** - Download all poses once calculations complete
4. **Analyze at scale** - Compare binding across all ligands

The `run()` method with `quote=True` first provides a cost estimate before submitting jobs. You can specify:
- **pocket**: Which binding pocket to use
- **batch_size**: How many ligands to process per batch



In [ ]:
jobs = sim.docking.run(
    pocket=pockets[0],
    quote=True,
    batch_size=8,
)
jobs

## Review Job Details

Before confirming, review the job details including:
- Number of ligands to dock
- Estimated cost
- Expected completion time



Use `confirm()` to submit the jobs for execution.


In [ ]:
jobs.confirm()
jobs

## Monitor Job Progress

The `watch()` method monitors your docking jobs and updates you on their progress. It will:
- Check job status at regular intervals
- Display progress updates
- Notify you when jobs complete

You can cancel jobs if needed using `jobs.cancel()`.


In [ ]:
jobs.watch()

## Retrieve Docking Results

Once jobs complete, retrieve all poses using `get_poses()`. This downloads all calculated poses for all ligands in your set.


In [ ]:
poses = sim.docking.get_poses()
poses

## Convert to DataFrame for Analysis

Convert poses to a DataFrame for detailed analysis. This enables:
- Statistical analysis of binding energies
- Comparison across ligands
- Filtering and sorting
- Export to CSV or other formats


In [ ]:
df = poses.to_dataframe()
df

## Visualize Statistics of All Poses

Create a scatter plot showing all poses from all docked ligands. The plot displays binding energy vs Pose Score. Hover over each point to see details about the ligand and pose.


In [ ]:
poses.plot()

## Visualize Statistics of Best Poses

Display the top pose for each ligand in the protein structure. This gives you a visual overview of how different ligands bind to the protein, helping you identify promising candidates for further study.



In [ ]:
top_poses = poses.filter_top_poses()
top_poses.plot()

## Show best poses

Find the best pose for each ligand and visualize their conformations in the protein structure. This helps identify the most promising binding modes across your ligand set.

In [ ]:
sim.protein.show(poses=top_poses)